## Airline AI Assistant


In [1]:
# Imports: `gradio` for building the chat UI and `OpenAI` for model client usage

import gradio as gr
from openai import OpenAI

d:\CodeBase\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# Start the Ollama server (shell magic). Run this in a notebook cell or terminal to launch the local model server.

!ollama serve

^C


In [17]:
# Quick health-check: verify the Ollama server is reachable before making requests
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [4]:
# Initialize the Ollama/OpenAI client pointing at the local server

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [5]:
# System prompt: defines assistant behavior, tone, and response constraints

system_prompt = """
You are a helpful customer service assistant for FlightAI airline.
Keep responses brief and courteous (1-2 sentences).
Use tools to find flight prices, times, and routes when asked about specific flights.
If a flight doesn't exist, inform the customer clearly.
Always be accurate. If unsure, say so.
"""

In [6]:
# Chat function: build message list, stream completions from Ollama, and yield incremental responses to the UI

def chat(user_message, chat_history):
    messages = (
        [{"role": "system", "content": system_prompt}]
        + chat_history
        + [{"role": "user", "content": user_message}]
    )

    with ollama.chat.completions.stream(
        model="ministral-3:3B", messages=messages) as stream:
        response = ""
        for chunk in stream:
            if chunk.type == "content.delta":
                response += chunk.delta or ""
                yield response  # yield only when new content arrives

In [7]:
# Launch the Gradio Chat UI with the streaming `chat` function

# gr.ChatInterface(fn=chat, title="Chatbot", description="Airline AI Assistant").launch(show_error=True)

## Tools
- tools are functions that the assistant can call to perform specific tasks. They are defined in the `tools` directory and can be imported into the main assistant code.
- Each tool should have a clear purpose and be designed to handle a specific task related to airline operations, such as booking flights, checking flight status, or providing customer support.
- The assistant can call these tools as needed to assist users with their inquiries and requests.

In [8]:
# Sample flights dataset used by the assistant tools (list of dicts)

flights = [
    {"flight_id": "FL-001", "origin_city": "Delhi", "destination_city": "Mumbai", "time": "08:00", "price": 5500},
    {"flight_id": "FL-002", "origin_city": "Mumbai", "destination_city": "Bangalore", "time": "10:30", "price": 4800},
    {"flight_id": "FL-003", "origin_city": "Delhi", "destination_city": "Kolkata", "time": "12:15", "price": 6200},
    {"flight_id": "FL-004", "origin_city": "Bangalore", "destination_city": "Chennai", "time": "14:45", "price": 3900},
    {"flight_id": "FL-005", "origin_city": "Mumbai", "destination_city": "Delhi", "time": "16:20", "price": 5500},
    {"flight_id": "FL-006", "origin_city": "Hyderabad", "destination_city": "Pune", "time": "09:00", "price": 3200},
    {"flight_id": "FL-007", "origin_city": "Kolkata", "destination_city": "Mumbai", "time": "11:30", "price": 6800},
    {"flight_id": "FL-008", "origin_city": "Chennai", "destination_city": "Delhi", "time": "13:45", "price": 7200},
    {"flight_id": "FL-009", "origin_city": "Pune", "destination_city": "Bangalore", "time": "15:00", "price": 4500},
    {"flight_id": "FL-010", "origin_city": "Hyderabad", "destination_city": "Delhi", "time": "17:30", "price": 5900}
]

In [9]:
# Step 1: Define tool functions that the assistant can call
# - `get_ticket_price`: find a flight by origin/destination and return price
# - `get_departure_time`: find a flight and return departure time
# - `get_flights_to_destination`: find flights to a specific destination and return details

def get_ticket_price(origin, destination):
    print(f"tool called: get_ticket_price({origin}, {destination})")
    for flight in flights:
        if flight["origin_city"].lower() == origin.lower() and flight["destination_city"].lower() == destination.lower():
            return f"The ticket price from {origin} to {destination} is ₹{flight['price']}."
    return f"No flight found from {origin} to {destination}."

def get_departure_time(origin, destination):
    print(f"tool called: get_departure_time({origin}, {destination})")
    for flight in flights:
        if flight["origin_city"].lower() == origin.lower() and flight["destination_city"].lower() == destination.lower():
            return f"The departure time from {origin} to {destination} is {flight['time']}."
    return f"No flight found from {origin} to {destination}."

def get_flights_to_destination(destination):
    print(f"tool called: get_flights_to_destination({destination})")
    flight_info = []
    for flight in flights:
        if flight["destination_city"].lower() == destination.lower():
            flight_info.append(f"Flight details for {destination}: Origin - {flight['origin_city']}, Destination - {flight['destination_city']}, Flight Name - {flight['flight_id']}, Time - {flight['time']}, Price - ₹{flight['price']}.")
    if flight_info:
        return "\n".join(flight_info)
    return f"No flight found to {destination}."

In [10]:
# Step 2: Tool metadata used for function-calling (e.g., for an LLM that supports calling typed tools)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the ticket price for a flight between two cities. Example: Delhi to Mumbai.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city"},
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["origin", "destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_departure_time",
            "description": "Get the departure time for a flight between two cities",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city"},
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["origin", "destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flights_to_destination",
            "description": "Get details of all flights to a specific destination city",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["destination"]
            }
        }
    }
]

In [11]:
# Step 3: Map tool names to the actual Python function objects so the assistant can call them

tool_functions = {
    "get_ticket_price": get_ticket_price,
    "get_departure_time": get_departure_time,
    "get_flights_to_destination": get_flights_to_destination
}

In [12]:
# Step 4: Handle tool calls from the model response (OpenAI/Ollama style)
import json

def handle_tool_calls(message):
    """
    Handle all tool calls from a model message and return a list of tool response messages.
    """
    if not message.tool_calls:
        return []

    tool_responses = []

    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        try:
            arguments = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}
        except json.JSONDecodeError:
            arguments = {}

        if tool_name in tool_functions:
            try:
                tool_result = tool_functions[tool_name](**arguments)
                content = str(tool_result)
            except Exception as e:
                content = f"Error executing {tool_name}: {str(e)}"
        else:
            content = f"Tool {tool_name} not found."

        tool_responses.append({
            "role": "tool",
            "name": tool_name,
            "content": content,
            "tool_call_id": tool_call.id
        })
    
    return tool_responses

In [13]:
# Step 5: Enhanced chat function with tool-calling support
# This version uses a single model for both generation and tool response processing

def chat_with_tools(user_message, chat_history):
    """
    Enhanced chat function that supports tool calling using a single model.
    The model can decide to call tools to gather flight information before responding.
    """
    # model_name = "qwen3.5:4b-q4_K_M"      # thinking model
    model_name = "qwen2.5:7b-instruct"
    # model_name = "llama3.2"

    messages = (
        [{"role": "system", "content": system_prompt}]
        + chat_history
        + [{"role": "user", "content": user_message}]
    )

    while True:
        response = ollama.chat.completions.create(
            model=model_name,
            messages=messages,
            extra_body={"think": False},  # Disable internal "thinking" to get immediate tool calls
            tools=tools,
        )

        assistant_message = response.choices[0].message  # message = {"role": "assistant", "content": "...", "tool_calls": [...]}
        messages.append(assistant_message)
        
        if assistant_message.tool_calls:
            tool_responses = handle_tool_calls(assistant_message)
            messages.extend(tool_responses)
            # Loop continues to let the model process tool results and provide a final answer
        else:
            # No more tool calls, return final content
            return assistant_message.content

In [ ]:
# Launch the Gradio Chat UI with the streaming `chat_with_tools` function

gr.ChatInterface(fn=chat_with_tools, title="Airline AI Assistant").launch(show_error=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


tool called: get_flights_to_destination(Delhi)
tool called: get_ticket_price(Hyderabad, Mumbai)
